## 0. One-time setup

In [23]:
# Install dependencies (run once). Restart the kernel after installing if prompted.
%pip install -q google-adk google-cloud-aiplatform[adk,agent_engines] google-cloud-storage google-genai requests

## 1. Initialize Vertex AI

In [24]:
import os

import vertexai
from google.cloud import storage
from vertexai.preview import reasoning_engines
from google.genai import types
from google.adk.models import Gemini
from google.adk.agents import Agent
from google.adk.tools import google_search

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
PROJECT_ID = "qwiklabs-gcp-03-8f57c8b00ccc"
LOCATION = os.getenv("GOOGLE_CLOUD_LOCATION", "us-central1")
STAGING_BUCKET_NAME = f"{PROJECT_ID}-agent-engine-staging"
STAGING_BUCKET = f"gs://{STAGING_BUCKET_NAME}"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# agent_engines.create() needs the staging bucket to already exist -- create
# it if this is the first run.
storage_client = storage.Client(project=PROJECT_ID)
if storage_client.lookup_bucket(STAGING_BUCKET_NAME) is None:
    print(f"Staging bucket {STAGING_BUCKET!r} not found -- creating it in {LOCATION!r}...")
    storage_client.create_bucket(STAGING_BUCKET_NAME, location=LOCATION)
else:
    print(f"Staging bucket {STAGING_BUCKET!r} already exists.")

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

MODEL_NAME = os.getenv("MODEL", "gemini-2.5-flash")

# Retry options help avoid the occasional error from popular models
# receiving too many requests at once.
RETRY_OPTIONS = types.HttpRetryOptions(initial_delay=1, max_delay=3, attempts=30)

print(f"Vertex AI initialized. PROJECT_ID={PROJECT_ID!r}, LOCATION={LOCATION!r}, STAGING_BUCKET={STAGING_BUCKET!r}")

Staging bucket 'gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging' not found -- creating it in 'us-central1'...
Vertex AI initialized. PROJECT_ID='qwiklabs-gcp-03-8f57c8b00ccc', LOCATION='us-central1', STAGING_BUCKET='gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging'


## 2. Define the agent

In [25]:
def build_search_agent() -> Agent:
    """Build a fresh search_agent instance.

    agent_engines.create() deepcopies the agent it's given, which fails with
    "cannot pickle '_thread.lock' object" if that agent (or its AdkApp) has
    already been used to run a query -- running a query opens live HTTP
    connections that can't be pickled. So the local test (Section 3) and the
    deploy step (Section 4) each get their own never-yet-run instance.
    """
    return Agent(
        name="search_agent",
        model=Gemini(model=MODEL_NAME, retry_options=RETRY_OPTIONS),
        description="You can search Google to answer questions.",
        instruction="""You are a helpful Chatbot. Use the google_search tool to answer the
user's question with current, factual information.""",
        tools=[google_search],
    )


search_agent = build_search_agent()
print("search_agent ready:", search_agent.name)

search_agent ready: search_agent


## 3. Test the agent locally

In [26]:
app = reasoning_engines.AdkApp(agent=search_agent)

for event in app.stream_query(
    user_id="local-test-user",
    message="What are the most popular books right now?",
):
    author = event.get("author", "?")
    for part in event.get("content", {}).get("parts", []):
        if part.get("text"):
            print(f"[{author}] {part['text'].strip()}")
        elif part.get("function_call"):
            fc = part["function_call"]
            print(f"[{author}] CALL {fc.get('name')}({fc.get('args')})")
        elif part.get("function_response"):
            fr = part["function_response"]
            print(f"[{author}] RESPONSE {fr.get('name')} -> {fr.get('response')}")

[search_agent] As of August 2026, several titles are making waves across bestseller lists and receiving significant attention from readers and critics alike.

**Popular Fiction Books Include:**

*   **"The Calamity Club" by Kathryn Stockett** is an instant *New York Times* bestseller. The author of "The Help" returns with a story about resilient women in 1933 Oxford, Mississippi, fighting for their rights and showcasing the power of friendship. It's also a popular book club pick.
*   **"Yesteryear: A GMA Book Club Pick" by Caro Claire Burke** follows a privileged influencer who awakens in 1855, grappling with a harsh new reality.
*   **"Whistler: A Novel" by Ann Patchett** explores a woman reconnecting with her stepfather decades later, reflecting on their past and divergent lives.
*   **"Cool Machine" by Colson Whitehead** is the third installment in his Harlem trilogy, delving into the impact of 1980s crime and capitalism.
*   **"Ransom: A Novel (Gabriel Allon #27)" by Daniel Silva**

## 4. Deploy the agent to Agent Engine

In [27]:
from vertexai import agent_engines

# A fresh Agent + AdkApp -- not the `app` from Section 3, which already ran a
# query and so holds unpicklable live connection state (see build_search_agent).
deploy_app = reasoning_engines.AdkApp(agent=build_search_agent())

remote_agent = agent_engines.create(
    deploy_app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
)

print("Deployed remote_agent:", remote_agent.resource_name)

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.162.0', 'pydantic': '2.13.4'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.13.4'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging
INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
I

Deployed remote_agent: projects/455595111103/locations/us-central1/reasoningEngines/5154943168839417856


## 5. Test the deployed agent

In [28]:
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="What are some popular movies right now?",
):
    author = event.get("author", "?")
    for part in event.get("content", {}).get("parts", []):
        if part.get("text"):
            print(f"[{author}] {part['text'].strip()}")
        elif part.get("function_call"):
            fc = part["function_call"]
            print(f"[{author}] CALL {fc.get('name')}({fc.get('args')})")
        elif part.get("function_response"):
            fr = part["function_response"]
            print(f"[{author}] RESPONSE {fr.get('name')} -> {fr.get('response')}")

[search_agent] Some of the most popular movies right now, based on their high box office performance in 2024, include "Inside Out 2," "Deadpool & Wolverine," and "Despicable Me 4".

Other top-grossing films of 2024 include:
*   "Dune: Part Two"
*   "Moana 2"
*   "Godzilla x Kong: The New Empire"
*   "Kung Fu Panda 4"
*   "Venom: The Last Dance"
*   "Wicked"
*   "Beetlejuice Beetlejuice"

"Inside Out 2" notably surpassed expectations, becoming the highest-grossing film of 2024 and achieving the second-biggest domestic opening ever for an animated movie.


![DeployedAgent](DeployedAgent.png)

## 6. Clean up

Deployed Agent Engine resources keep running (and billing) until deleted. Uncomment and run this
cell when you're done testing.

In [29]:
TEST_USER_IDS = ["agent-engine-test-user"]

# Delete Sessions
sessions_to_delete = []
for user_id in TEST_USER_IDS:
    sessions_page = remote_agent.list_sessions(user_id=user_id)
    for session in sessions_page.get("sessions", []):
        sessions_to_delete.append((user_id, session["id"]))

print(f"Found {len(sessions_to_delete)} session(s) to delete.")

for user_id, session_id in sessions_to_delete:
    remote_agent.delete_session(user_id=user_id, session_id=session_id)
    print(f"Deleted session {session_id!r} (user_id={user_id!r})")

# Delete agent
remote_agent.delete()
print("Deleted remote_agent:", remote_agent.resource_name)

# Delete bucket
bucket = storage_client.bucket(STAGING_BUCKET_NAME)
blobs_to_delete = list(bucket.list_blobs())
print(f"Found {len(blobs_to_delete)} object(s) to delete in {STAGING_BUCKET!r}.")
for blob in blobs_to_delete:
    blob.delete()
    print(f"Deleted object {blob.name!r}")
bucket.delete()
print(f"Deleted staging bucket {STAGING_BUCKET!r}")

Found 1 session(s) to delete.


INFO:vertexai.agent_engines:Delete Agent Engine backing LRO: projects/455595111103/locations/us-central1/operations/6710804250160529408
INFO:vertexai.agent_engines:Agent Engine deleted. Resource name: projects/455595111103/locations/us-central1/reasoningEngines/5154943168839417856


Deleted session '6381922210415640576' (user_id='agent-engine-test-user')
Deleted remote_agent: projects/455595111103/locations/us-central1/reasoningEngines/5154943168839417856
Found 3 object(s) to delete in 'gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging'.
Deleted object 'agent_engine/agent_engine.pkl'
Deleted object 'agent_engine/dependencies.tar.gz'
Deleted object 'agent_engine/requirements.txt'
Deleted staging bucket 'gs://qwiklabs-gcp-03-8f57c8b00ccc-agent-engine-staging'
